# 개별종목 조합D — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.5057,0.5012,0.0045,0.2828,0.0556,0.1277
1,2,NaN,980,20150123,20150421,0.4079,0.3978,0.0101,0.3111,0.0935,0.1834
2,3,NaN,1210,20151228,20160328,0.3820,0.3762,0.0058,0.3296,0.1777,0.2660
3,4,balanced,1439,20161202,20170228,0.4547,0.4617,-0.0070,0.3571,0.1317,0.2382
4,5,balanced,1669,20171113,20180207,0.4018,0.3901,0.0118,0.3693,0.2044,0.2973
5,6,balanced,1899,20181024,20190118,0.3963,0.3725,0.0238,0.3808,0.2296,0.3156
6,7,balanced,2129,20190930,20191224,0.4645,0.4781,-0.0137,0.3512,0.1257,0.2316
7,8,NaN,2359,20200902,20201130,0.3690,0.3476,0.0214,0.3586,0.3193,0.3476
8,9,NaN,2589,20210806,20211105,0.4201,0.3916,0.0285,0.3669,0.2260,0.3148
9,10,NaN,2818,20220714,20221012,0.3634,0.3454,0.0179,0.3285,0.2062,0.2818


,OOS 폴드 평균
accuracy,0.4117
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0148
macro_f1,0.3490
down_recall,0.1960
core_harmonic_mean,0.2745


재실행 명령: python scripts/run_stock_model_experiment.py
